# CozyDay Multi-Model AI System

P1-A7 Part 1 | Team 3 | Stage 1 Lin

Stage 1 defines all 15 models and 25 prompts and reports Models 1–5. Later stages supply Models 6–15 and final routing/analysis. Run All reads committed results and recomputes costs in memory; it never runs inference, downloads weights, or writes authoritative CSVs.

## Contents

1. [Setup and model categories](#setup-and-model-categories)
2. [Prompts](#prompts)
3. [Evaluation rubric](#evaluation-rubric)
4. [Saved benchmarks](#saved-benchmarks)
5. [Scaling and costs](#scaling-and-costs)
6. [Team handoff](#team-handoff)

Run from the repository root or `llm-test`. Results-only dependencies: pandas and IPython (plus a Jupyter kernel to execute this notebook). Optional inference additionally needs torch, transformers, accelerate, and bitsandbytes for NF4. No optional inference library is imported on Run All.


## Setup and model categories

The unchanged A6 categories use nominal parameter labels: Tiny <0.5B; Small 0.5B–1.5B inclusive; Medium >1.5B–3B; Large >3B–7B; Very Large >7B. Each category contains three models. Exact tensor parameter counts can differ from nominal model names; this is a grouping convention, not an exact parameter-count audit.


In [1]:
import json
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
MODULE_DIR = next((d for d in (here, here / "llm-test")
                   if (d / "a7_shared.py").is_file()), None)
if MODULE_DIR is None:
    raise FileNotFoundError("Run from the repo root or llm-test directory")
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
from a7_shared import (registry, validate_prompts, validate_records, run_models,
                       calculate_costs, SCORE_COLUMNS)
RESULTS_DIR = MODULE_DIR / "results"
model_matrix = pd.read_csv(RESULTS_DIR / "research_matrix.csv")
models = registry(model_matrix)
assert model_matrix["Category"].nunique() == 5
assert model_matrix["Category"].value_counts().eq(3).all()
display(model_matrix[["Model No.", "Model Name", "Category", "Parameters"]])
print("15 models; five categories; three models per category.")


,Model No.,Model Name,Category,Parameters
0,1,HuggingFaceTB/SmolLM2-135M-Instruct,Tiny (<0.5B),135M
1,2,Qwen/Qwen2.5-0.5B-Instruct,Small (0.5B-1.5B),0.5B
2,3,Qwen/Qwen3-1.7B,Medium (>1.5B-3B),1.7B
3,4,microsoft/Phi-3.5-mini-instruct,Large (>3B-7B),3.8B
4,5,Qwen/Qwen3-8B,Very Large (>7B),8.2B
5,6,HuggingFaceTB/SmolLM-135M-Instruct,Tiny (<0.5B),135M
6,7,Qwen/Qwen3-0.6B,Small (0.5B-1.5B),0.6B
7,8,google/gemma-2-2b-it,Medium (>1.5B-3B),~2B
8,9,Qwen/Qwen3-4B,Large (>3B-7B),4B
9,10,mistralai/Mistral-Nemo-Instruct-2407,Very Large (>7B),12B


15 models; five categories; three models per category.


## Prompts

The five CozyDay types are Task Classification, Priority Ranking, Schedule Generation, Plan Summarization, and Structured Extraction. The following 25 prompts and IDs are unchanged from the measured experiments.


In [2]:

import pandas as pd

evaluation_prompts = [
    # Task Classification
    ("C1", "Task Classification", "Classify these tasks as Work, Study, Personal, or Health: finish project report, review lecture notes, buy groceries, and go for a run. Return one category per task."),
    ("C2", "Task Classification", "Categorize my tasks: email my supervisor, prepare for an exam, clean my room, and attend a dental appointment."),
    ("C3", "Task Classification", "Classify these activities: submit homework, schedule a team meeting, call my family, and attend a yoga class."),
    ("C4", "Task Classification", "Organize these tasks by category: write a research summary, cook dinner, review class slides, and visit the gym."),
    ("C5", "Task Classification", "Classify these tasks as Work, Study, Personal, or Health: update a client spreadsheet, study Python, pay rent, and attend a medical checkup."),

    # Priority Ranking
    ("P1", "Priority Ranking", "I have a report due tomorrow, groceries to buy, and a room to clean. Rank these tasks by priority and explain briefly."),
    ("P2", "Priority Ranking", "I have an exam tomorrow, a meeting next week, and laundry to finish. What should I prioritize today?"),
    ("P3", "Priority Ranking", "My presentation is due tonight, my emails can wait until Friday, and I need to exercise. Rank these tasks."),
    ("P4", "Priority Ranking", "I must submit an assignment by 5 PM, call a friend sometime this week, and prepare dinner tonight. Rank these tasks by urgency."),
    ("P5", "Priority Ranking", "I need to finish a group project due tomorrow, organize my desk, and prepare for a quiz next Monday. Give me a priority order."),

    # Schedule Generation
    ("S1", "Schedule Generation", "Create a schedule from 9 AM to 5 PM for studying 3 hours, exercising 1 hour, and grocery shopping 1 hour. Include breaks."),
    ("S2", "Schedule Generation", "Plan my day between 8 AM and 6 PM. I have class from 10 AM to noon, need 2 hours for homework, and want 1 hour for lunch."),
    ("S3", "Schedule Generation", "I am free from 1 PM to 9 PM. Schedule 3 hours of project work, 1 hour of exercise, and 1 hour of cooking."),
    ("S4", "Schedule Generation", "Build a realistic schedule from 9 AM to 7 PM with a meeting from 11 AM to noon, 3 hours of studying, and 2 hours of errands."),
    ("S5", "Schedule Generation", "Organize my afternoon from noon to 6 PM. I need 2 hours for research, 1 hour for groceries, and 1 hour for cleaning."),

    # Plan Summarization
    ("M1", "Plan Summarization", "Summarize my day: 9 AM lecture, 11 AM library study, 2 PM grocery shopping, and 6 PM dinner with friends."),
    ("M2", "Plan Summarization", "Give me a concise summary of tomorrow: morning workout, afternoon team meeting, evening homework."),
    ("M3", "Plan Summarization", "Summarize this schedule and highlight key commitments: 8 AM breakfast, 10 AM exam, 1 PM lunch, 3 PM office hours."),
    ("M4", "Plan Summarization", "Summarize my tasks in two sentences: finish readings, send emails, clean the apartment, and prepare meals."),
    ("M5", "Plan Summarization", "Provide a short daily overview: 9 AM project work, noon lunch, 2 PM class, 5 PM gym, and 7 PM dinner."),

    # Structured Extraction
    ("E1", "Structured Extraction", "Extract tasks and times as JSON from: I need to attend class at 9 AM, buy groceries at 4 PM, and study at 7 PM."),
    ("E2", "Structured Extraction", "Convert this into JSON with task and deadline fields: Finish the report by Friday, email the professor by Wednesday, and submit homework tomorrow."),
    ("E3", "Structured Extraction", "Extract tasks, durations, and categories into JSON: study Python for 2 hours, exercise for 45 minutes, and clean the kitchen for 30 minutes."),
    ("E4", "Structured Extraction", "Create a JSON task list from: Meeting at 10 AM, lunch at noon, project work at 2 PM, and dinner at 6 PM."),
    ("E5", "Structured Extraction", "Extract structured task data as JSON: submit the assignment by 11:59 PM, call my parents at 7 PM, and exercise for 30 minutes.")
]

prompts_df = pd.DataFrame(
    evaluation_prompts,
    columns=["Prompt ID", "Prompt Type", "Prompt"]
)

display(prompts_df)

assert len(prompts_df) == 25
assert prompts_df["Prompt Type"].nunique() == 5
assert all(prompts_df["Prompt Type"].value_counts() == 5)

print("Total prompts:", len(prompts_df))
print("Prompt types:", prompts_df["Prompt Type"].nunique())
print(prompts_df["Prompt Type"].value_counts())

validate_prompts(prompts_df)


,Prompt ID,Prompt Type,Prompt
0,C1,Task Classification,"Classify these tasks as Work, Study, Personal,..."
1,C2,Task Classification,"Categorize my tasks: email my supervisor, prep..."
2,C3,Task Classification,"Classify these activities: submit homework, sc..."
3,C4,Task Classification,Organize these tasks by category: write a rese...
4,C5,Task Classification,"Classify these tasks as Work, Study, Personal,..."
5,P1,Priority Ranking,"I have a report due tomorrow, groceries to buy..."
6,P2,Priority Ranking,"I have an exam tomorrow, a meeting next week, ..."
7,P3,Priority Ranking,"My presentation is due tonight, my emails can ..."
8,P4,Priority Ranking,"I must submit an assignment by 5 PM, call a fr..."
9,P5,Priority Ranking,"I need to finish a group project due tomorrow,..."


Total prompts: 25
Prompt types: 5
Prompt Type
Task Classification      5
Priority Ranking         5
Schedule Generation      5
Plan Summarization       5
Structured Extraction    5
Name: count, dtype: int64


## Evaluation rubric

Four integer criteria receive equal weight: instruction adherence, accuracy, clarity, and structure. Each is scored 1–10: 1–2 very poor; 3–4 poor; 5–6 acceptable; 7–8 good; 9–10 excellent. Overall score is the mean rounded half up. Scores are assigned from actual responses; inference failures receive no invented score.

Classification must cover every task (C2–C4 do not mandate a fixed label vocabulary). Prioritization should respect explicit deadlines without inventing urgency. Schedules require correct durations and no collisions. Summaries must preserve supplied facts and sentence-count instructions. Extraction must include requested tasks, avoid inventions and use JSON; Markdown code fences are accepted if their contents parse. Truncation/omissions reduce adherence or structure. These are subjective human scores, not an independent quality benchmark.

The reviewed calibration is preserved: Model 2 P5 = 8; Model 3 M4 = 8; Model 4 S5 = 9; Model 5 S1 = 8. Model 4 S5 has accuracy 6 for its incorrect buffer arithmetic; its 8.5 criterion mean still rounds to 9. Full rationales are in system_design.md.


In [3]:
import math
import pandas as pd

EVALUATION_METRICS = [
    "Instruction Adherence",
    "Accuracy",
    "Clarity",
    "Structure"
]

def calculate_score(instruction, accuracy, clarity, structure):
    scores = [instruction, accuracy, clarity, structure]

    if not all(isinstance(x, int) and 1 <= x <= 10 for x in scores):
        raise ValueError("Each metric must be an integer between 1 and 10.")

    average = sum(scores) / len(scores)

    return math.floor(average + 0.5)

# Verify scoring function
assert calculate_score(8, 9, 7, 8) == 8
assert calculate_score(9, 9, 10, 10) == 10

print("Evaluation metrics:", EVALUATION_METRICS)
print("Example overall score:", calculate_score(8, 9, 7, 8))
print("Scoring system ready!")


Evaluation metrics: ['Instruction Adherence', 'Accuracy', 'Clarity', 'Structure']
Example overall score: 8
Scoring system ready!


## Saved benchmarks

All five raw benchmark files, five scored files, combined 125-row file and summary are loaded without inference or writes. Latency is generate-call seconds (excludes model loading, chat formatting, and decoding). Throughput is generated token IDs divided by generation seconds, including any generated special tokens. Stored values are rounded to three decimals. One run per prompt, no standardized warm-up or repeat-trial uncertainty estimate.

Models 1–3: local CPU bfloat16 metadata; Model 4: Colab T4 FP16; Model 5: Colab T4 NF4, double quantization, FP16 compute, thinking disabled. Hardware/precision differences prevent attributing speed changes to size alone. Historical GPU synchronization is reported by the experiment author; original GPU execution logs are not stored here. CPU model/RAM/thread settings and exact model revisions were not recorded, so exact historical timing reproduction is limited.


In [4]:
frames = []
for number in range(1, 6):
    raw = pd.read_csv(RESULTS_DIR / f"a7_model_{number}_benchmark.csv")
    scored = pd.read_csv(RESULTS_DIR / f"a7_model_{number}_scored.csv")
    validate_records(scored, *models[number], prompts_df)
    assert len(scored) == 25
    unscored = SCORE_COLUMNS + ["Self-Evaluation Score (1-10)"]
    pd.testing.assert_frame_equal(raw.drop(columns=unscored), scored.drop(columns=unscored))
    assert scored[SCORE_COLUMNS].map(lambda x: float(x).is_integer() and 1 <= x <= 10).all().all()
    calculated = scored[SCORE_COLUMNS].mean(axis=1).map(lambda x: int(x + 0.5))
    assert calculated.eq(scored["Self-Evaluation Score (1-10)"]).all()
    frames.append(scored.assign(**{"Model No.": number}))
stage1_benchmark_df = pd.concat(frames, ignore_index=True)
saved_combined = pd.read_csv(RESULTS_DIR / "a7_stage1_all_benchmarks.csv")
pd.testing.assert_frame_equal(stage1_benchmark_df[saved_combined.columns], saved_combined)
stage1_summary = stage1_benchmark_df.groupby(["Model No.", "Model", "Model Category"]).agg(
    Prompts=("Prompt ID", "count"), Average_Score=("Self-Evaluation Score (1-10)", "mean"),
    Average_Latency=("Latency", "mean"), Average_Tokens_Per_Sec=("Tokens/sec", "mean"),
    Token_Limit_Hits=("Reached Token Limit", "sum")).reset_index().round(2)
pd.testing.assert_frame_equal(stage1_summary, pd.read_csv(RESULTS_DIR / "a7_stage1_summary.csv"))
display(stage1_summary)
with pd.option_context("display.max_rows", 150, "display.max_colwidth", None):
    display(stage1_benchmark_df[["Model No.", "Prompt ID", "Model Category", "Tokens/sec",
        "Latency", "Prompt Type", "Prompt", "Self-Evaluation Score (1-10)"]])
print("PASS: all 125 scored records, prompt text, measurements and score arithmetic.")


,Model No.,Model,Model Category,Prompts,Average_Score,Average_Latency,Average_Tokens_Per_Sec,Token_Limit_Hits
0,1,HuggingFaceTB/SmolLM2-135M-Instruct,Tiny (<0.5B),25,2.76,5.19,24.49,10
1,2,Qwen/Qwen2.5-0.5B-Instruct,Small (0.5B-1.5B),25,6.76,5.56,12.37,1
2,3,Qwen/Qwen3-1.7B,Medium (>1.5B-3B),25,7.96,17.34,4.34,2
3,4,microsoft/Phi-3.5-mini-instruct,Large (>3B-7B),25,9.40,6.05,24.21,3
4,5,Qwen/Qwen3-8B,Very Large (>7B),25,9.60,6.66,11.76,1


,Model No.,Prompt ID,Model Category,Tokens/sec,Latency,Prompt Type,Prompt,Self-Evaluation Score (1-10)
0,1,C1,Tiny (<0.5B),16.163,1.485,Task Classification,"Classify these tasks as Work, Study, Personal, or Health: finish project report, review lecture notes, buy groceries, and go for a run. Return one category per task.",4
1,1,C2,Tiny (<0.5B),35.017,7.311,Task Classification,"Categorize my tasks: email my supervisor, prepare for an exam, clean my room, and attend a dental appointment.",2
2,1,C3,Tiny (<0.5B),28.313,1.483,Task Classification,"Classify these activities: submit homework, schedule a team meeting, call my family, and attend a yoga class.",2
3,1,C4,Tiny (<0.5B),23.979,1.126,Task Classification,"Organize these tasks by category: write a research summary, cook dinner, review class slides, and visit the gym.",2
4,1,C5,Tiny (<0.5B),21.011,0.999,Task Classification,"Classify these tasks as Work, Study, Personal, or Health: update a client spreadsheet, study Python, pay rent, and attend a medical checkup.",3
5,1,P1,Tiny (<0.5B),27.671,9.251,Priority Ranking,"I have a report due tomorrow, groceries to buy, and a room to clean. Rank these tasks by priority and explain briefly.",1
6,1,P2,Tiny (<0.5B),27.171,9.238,Priority Ranking,"I have an exam tomorrow, a meeting next week, and laundry to finish. What should I prioritize today?",4
7,1,P3,Tiny (<0.5B),27.445,2.405,Priority Ranking,"My presentation is due tonight, my emails can wait until Friday, and I need to exercise. Rank these tasks.",4
8,1,P4,Tiny (<0.5B),22.984,1.436,Priority Ranking,"I must submit an assignment by 5 PM, call a friend sometime this week, and prepare dinner tonight. Rank these tasks by urgency.",2
9,1,P5,Tiny (<0.5B),29.143,2.985,Priority Ranking,"I need to finish a group project due tomorrow, organize my desk, and prepare for a quiz next Monday. Give me a priority order.",2


PASS: all 125 scored records, prompt text, measurements and score arithmetic.


## Scaling and costs

Prospective workload: 250 input + 500 generated tokens/request, one request/DAU/day, 30 days/month. The 500-output-token assumption is a planning stress scenario, not the benchmark mean or its 256-token cap. Cloud inference capacity is assumed at 60/45/30/20/12 output tokens/sec for Models 1–5 and 70% utilization. Reference cloud machine: AWS g5.xlarge, A10G 24 GB; USD 1.006/hour is an unverified planning price assumption, not a newly verified quote. CPU/RAM specs and selected model precision must be checked before provisioning. Gateway assumption: USD 0.05/hour and 100 requests/sec/machine, average-load sizing only.

Daily output load = DAU × requests/user/day × output tokens/request. Local machines = ceil(daily output / (tokens/sec × 86400 × utilization)). Local monthly cost = machines × hourly rate × 24 × 30. API monthly provider cost = 30 × (daily input × input USD/1M + daily output × output USD/1M) / 1,000,000. Round the provider bill and gateway bill half up to cents, then add them.

All 30 API scenarios now use verified HF listing prices. Models 1–4 share Qwen3-4B-Instruct-2507/nscale as a task-based instruct-model proxy at USD 0.01 input / 0.03 output per million tokens. Tiny and Small comparisons deliberately use larger paid models, not equal-size or equal-quality substitutes. Model 5 uses Qwen3-8B/nscale at USD 0.07 input / 0.18 output per million. Checked 2026-10-21 against https://huggingface.co/inference/models and the official Nscale-supported model list. The unsupported DeepSeek-1.5B/nscale pairing and old hypothetical prices are removed. Source snapshots and comparison notes are in a7_api_pricing.json; prices and availability can change.

Local per-machine price is USD/hour; total machine cost is USD/month. Token loads are daily output tokens (API includes a separate daily input field). The shared functions return tables without saving. Gateway machines are client-side gateways, not estimates of the provider's GPU fleet. Input prefill cost affects cloud throughput but is not separately modeled in the local capacity assumption. Storage, network, redundancy, labor, taxes, discounts, provider quotas and peak-load queues are excluded. No paid API inference or quality test was performed.


In [5]:
pricing = json.loads((MODULE_DIR / "a7_api_pricing.json").read_text())
api_models = {int(k): v for k, v in pricing["models"].items()}
cloud_throughput = {1: 60, 2: 45, 3: 30, 4: 20, 5: 12}
local_cost_df, api_cost_df = calculate_costs(range(1, 6), model_matrix, cloud_throughput, api_models)
saved_local = pd.read_csv(RESULTS_DIR / "a7_stage1_local_cost.csv")
saved_api = pd.read_csv(RESULTS_DIR / "a7_stage1_api_cost.csv")
pd.testing.assert_frame_equal(local_cost_df, saved_local, check_dtype=False)
pd.testing.assert_frame_equal(api_cost_df, saved_api, check_dtype=False)
assert len(local_cost_df) == len(api_cost_df) == 30
assert api_cost_df[["Estimated API Cost", "Total Cost"]].notna().all().all()
with pd.option_context("display.max_rows", 60, "display.max_columns", None):
    display(local_cost_df)
    display(api_cost_df)
print("PASS: 30 local + 30 sourced API scenarios reproduced without writing CSVs.")


,Model Name,Daily Traffic Category,Daily Active Users,Avg Tokens per User,Total Daily Token Load,Per Machine Cost (Cloud),Machines Needed,Total Machine Cost,Cloud Throughput Assumption (tokens/sec),Cost Period
0,HuggingFaceTB/SmolLM2-135M-Instruct,Prototype,1000,500,500000,1.006,1,724.32,60,USD/month
1,HuggingFaceTB/SmolLM2-135M-Instruct,Early Startup,10000,500,5000000,1.006,2,1448.64,60,USD/month
2,HuggingFaceTB/SmolLM2-135M-Instruct,Growing Product,100000,500,50000000,1.006,14,10140.48,60,USD/month
3,HuggingFaceTB/SmolLM2-135M-Instruct,Large Platform,1000000,500,500000000,1.006,138,99956.16,60,USD/month
4,HuggingFaceTB/SmolLM2-135M-Instruct,Mass Consumer App,10000000,500,5000000000,1.006,1378,998112.96,60,USD/month
5,HuggingFaceTB/SmolLM2-135M-Instruct,Global Platform,100000000,500,50000000000,1.006,13779,9980405.28,60,USD/month
6,Qwen/Qwen2.5-0.5B-Instruct,Prototype,1000,500,500000,1.006,1,724.32,45,USD/month
7,Qwen/Qwen2.5-0.5B-Instruct,Early Startup,10000,500,5000000,1.006,2,1448.64,45,USD/month
8,Qwen/Qwen2.5-0.5B-Instruct,Growing Product,100000,500,50000000,1.006,19,13762.08,45,USD/month
9,Qwen/Qwen2.5-0.5B-Instruct,Large Platform,1000000,500,500000000,1.006,184,133274.88,45,USD/month


,Local LLM Model Size,Comparable Paid Model Hosted on Hugging Face,Output USD per Token,Daily Traffic Category,Total Token Load,Estimated API Cost,Machines Required (API infrastructure),Total Cost,API Gateway Cost,Pricing Status,Cost Period,Paid Provider,Input USD per 1M tokens,Output USD per 1M tokens,Assumed Input Tokens per Request,Assumed Output Tokens per Request,Total Daily Input Tokens,Model Comparison Note,Price Source URL,Price Checked Date
0,HuggingFaceTB/SmolLM2-135M-Instruct,Qwen/Qwen3-4B-Instruct-2507,3.000000e-08,Prototype,500000,0.53,1,36.53,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.01,0.03,250,500,250000,Task-based 4B instruct proxy for a 135M local ...,https://huggingface.co/inference/models?model=...,2026-10-21
1,HuggingFaceTB/SmolLM2-135M-Instruct,Qwen/Qwen3-4B-Instruct-2507,3.000000e-08,Early Startup,5000000,5.25,1,41.25,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.01,0.03,250,500,2500000,Task-based 4B instruct proxy for a 135M local ...,https://huggingface.co/inference/models?model=...,2026-10-21
2,HuggingFaceTB/SmolLM2-135M-Instruct,Qwen/Qwen3-4B-Instruct-2507,3.000000e-08,Growing Product,50000000,52.50,1,88.50,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.01,0.03,250,500,25000000,Task-based 4B instruct proxy for a 135M local ...,https://huggingface.co/inference/models?model=...,2026-10-21
3,HuggingFaceTB/SmolLM2-135M-Instruct,Qwen/Qwen3-4B-Instruct-2507,3.000000e-08,Large Platform,500000000,525.00,1,561.00,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.01,0.03,250,500,250000000,Task-based 4B instruct proxy for a 135M local ...,https://huggingface.co/inference/models?model=...,2026-10-21
4,HuggingFaceTB/SmolLM2-135M-Instruct,Qwen/Qwen3-4B-Instruct-2507,3.000000e-08,Mass Consumer App,5000000000,5250.00,2,5322.00,72.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.01,0.03,250,500,2500000000,Task-based 4B instruct proxy for a 135M local ...,https://huggingface.co/inference/models?model=...,2026-10-21
5,HuggingFaceTB/SmolLM2-135M-Instruct,Qwen/Qwen3-4B-Instruct-2507,3.000000e-08,Global Platform,50000000000,52500.00,12,52932.00,432.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.01,0.03,250,500,25000000000,Task-based 4B instruct proxy for a 135M local ...,https://huggingface.co/inference/models?model=...,2026-10-21
6,Qwen/Qwen2.5-0.5B-Instruct,Qwen/Qwen3-4B-Instruct-2507,3.000000e-08,Prototype,500000,0.53,1,36.53,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.01,0.03,250,500,250000,Task-based 4B instruct proxy for a nominal 0.5...,https://huggingface.co/inference/models?model=...,2026-10-21
7,Qwen/Qwen2.5-0.5B-Instruct,Qwen/Qwen3-4B-Instruct-2507,3.000000e-08,Early Startup,5000000,5.25,1,41.25,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.01,0.03,250,500,2500000,Task-based 4B instruct proxy for a nominal 0.5...,https://huggingface.co/inference/models?model=...,2026-10-21
8,Qwen/Qwen2.5-0.5B-Instruct,Qwen/Qwen3-4B-Instruct-2507,3.000000e-08,Growing Product,50000000,52.50,1,88.50,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.01,0.03,250,500,25000000,Task-based 4B instruct proxy for a nominal 0.5...,https://huggingface.co/inference/models?model=...,2026-10-21
9,Qwen/Qwen2.5-0.5B-Instruct,Qwen/Qwen3-4B-Instruct-2507,3.000000e-08,Large Platform,500000000,525.00,1,561.00,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.01,0.03,250,500,250000000,Task-based 4B instruct proxy for a nominal 0.5...,https://huggingface.co/inference/models?model=...,2026-10-21


PASS: 30 local + 30 sourced API scenarios reproduced without writing CSVs.


## Team handoff

Rishabh owns Models 6–10; Shivani owns Models 11–15. Ashley owns final cross-model analysis, final local/API switching conclusions, integration and her routing scenarios. Their experiments are intentionally not executed or filled here.

Use registry(model_matrix), prepare_messages, run_models and calculate_costs from a7_shared.py. run_models defaults to execute=False; import, dry runs, and Run All do not load torch/transformers or download weights. Gemma 2 receives the system instruction prepended to a single user message; Qwen3 uses enable_thinking=False. Explicit GPU options are backend='cuda', precision='fp16' or 'nf4'. Historical CPU precision is 'bfloat16'; the default 'float32' is a new experiment configuration.

Completed historical records are validated and read unchanged, even if the requested new-run configuration differs: no inference occurs for completed models. Partial legacy files cannot be resumed into a new protocol. New partial runs require matching protocol, precision, backend and runtime metadata; use a fresh results directory for changed hardware/software. Atomic writes apply only to explicitly enabled new benchmark output files; scored files are never overwritten. Failures raise and retain prior completed rows, with no fabricated scores.

Example future inference call (documentation only): `run_models([6, 7], model_matrix, prompts_df, fresh_result_dir, execute=True, backend='cpu', precision='float32')`. Gemma requires license acceptance and HF authentication outside Git; 12B/14B GPU models may need NF4. Add verified provider metadata and throughput assumptions for each assigned model before calling calculate_costs. Save only new stage-specific tables deliberately; do not regenerate Stage 1 files. No weights/secrets belong in Git.


In [6]:
# Safe interface check only: no model inference or file writes.
assert run_models(range(6, 11), model_matrix, prompts_df, RESULTS_DIR, execute=False) == {}
print("Stage 2/3 registry and helpers available; inference disabled.")


Stage 2/3 registry and helpers available; inference disabled.


## Stage 2 — Models 6-10 (Rishabh)

Completed, real benchmark records for Models 6-10 (same 25 shared prompts, same shared rubric, same `a7_shared.py` helpers as Stage 1). Models 6-8 (`HuggingFaceTB/SmolLM-135M-Instruct`, `Qwen/Qwen3-0.6B`, `google/gemma-2-2b-it`) ran on local CPU, `backend="cpu"`, `precision="bfloat16"` (Stage 1's historical CPU convention; `a7_shared.load_model` has no MPS path, so these ran on CPU only, not Apple Silicon's GPU — Model 8 in particular paid for that in latency). Models 9-10 (`Qwen/Qwen3-4B`, `mistralai/Mistral-Nemo-Instruct-2407`) ran on a free-tier Colab T4 GPU: Model 9 at `precision="fp16"`, Model 10 at `precision="nf4"` (4-bit, `bitsandbytes`), matching Stage 1's own Large/Very Large GPU precision choices. Qwen3 thinking is disabled for Models 7 and 9 via `a7_shared.py`'s unconditional `enable_thinking=False` for any Qwen3 model id — the same mechanism Stage 1 used for Model 5 — and that setting is bound into each record's `protocol_id` hash, so it cannot silently vary between runs. Model 8's Gemma-2 chat template receives the system instruction prepended into a single user turn, the same wrapper Stage 1 established, since Gemma 2's own template rejects a system role. All five files were produced by `run_models(..., execute=True)` and pass `validate_records` unchanged from this notebook.

125 scored records live in `llm-test/results/a7_model_6_scored.csv` through `a7_model_10_scored.csv`. Scoring followed Section 4's rubric exactly: each of the 25 responses per model was read in full against the actual prompt and scored 1-10 on Instruction Adherence, Accuracy, Clarity and Structure, with the overall score `floor(mean + 0.5)`, same as Stage 1. No invented scores; a documented failure (an empty/garbled response, a duration error, a format miss) is scored low rather than skipped.


In [7]:
frames = []
for number in range(6, 11):
    raw = pd.read_csv(RESULTS_DIR / f"a7_model_{number}_benchmark.csv")
    scored = pd.read_csv(RESULTS_DIR / f"a7_model_{number}_scored.csv")
    validate_records(scored, *models[number], prompts_df)
    assert len(scored) == 25
    unscored = SCORE_COLUMNS + ["Self-Evaluation Score (1-10)"]
    pd.testing.assert_frame_equal(raw.drop(columns=unscored), scored.drop(columns=unscored))
    assert scored[SCORE_COLUMNS].map(lambda x: float(x).is_integer() and 1 <= x <= 10).all().all()
    calculated = scored[SCORE_COLUMNS].mean(axis=1).map(lambda x: int(x + 0.5))
    assert calculated.eq(scored["Self-Evaluation Score (1-10)"]).all()
    frames.append(scored.assign(**{"Model No.": number}))
stage2_benchmark_df = pd.concat(frames, ignore_index=True)
saved_combined_s2 = pd.read_csv(RESULTS_DIR / "a7_stage2_all_benchmarks.csv")
pd.testing.assert_frame_equal(stage2_benchmark_df[saved_combined_s2.columns], saved_combined_s2)
stage2_summary = stage2_benchmark_df.groupby(["Model No.", "Model", "Model Category"]).agg(
    Prompts=("Prompt ID", "count"), Average_Score=("Self-Evaluation Score (1-10)", "mean"),
    Average_Latency=("Latency", "mean"), Average_Tokens_Per_Sec=("Tokens/sec", "mean"),
    Token_Limit_Hits=("Reached Token Limit", "sum")).reset_index().round(2)
pd.testing.assert_frame_equal(stage2_summary, pd.read_csv(RESULTS_DIR / "a7_stage2_summary.csv"))
display(stage2_summary)
with pd.option_context("display.max_rows", 150, "display.max_colwidth", None):
    display(stage2_benchmark_df[["Model No.", "Prompt ID", "Model Category", "Tokens/sec",
        "Latency", "Prompt Type", "Prompt", "Self-Evaluation Score (1-10)"]])
print("PASS: all 125 Stage 2 scored records, prompt text, measurements and score arithmetic.")
print("Grand total:", int(stage2_benchmark_df["Self-Evaluation Score (1-10)"].sum()), "/ 1250")


,Model No.,Model,Model Category,Prompts,Average_Score,Average_Latency,Average_Tokens_Per_Sec,Token_Limit_Hits
0,6,HuggingFaceTB/SmolLM-135M-Instruct,Tiny (<0.5B),25,1.88,2.77,85.79,22
1,7,Qwen/Qwen3-0.6B,Small (0.5B-1.5B),25,6.16,2.02,21.86,0
2,8,google/gemma-2-2b-it,Medium (>1.5B-3B),25,8.04,615.87,0.16,0
3,9,Qwen/Qwen3-4B,Large (>3B-7B),25,8.40,4.30,18.02,1
4,10,mistralai/Mistral-Nemo-Instruct-2407,Very Large (>7B),25,8.44,5.51,12.46,0


,Model No.,Prompt ID,Model Category,Tokens/sec,Latency,Prompt Type,Prompt,Self-Evaluation Score (1-10)
0,6,C1,Tiny (<0.5B),79.613,3.216,Task Classification,"Classify these tasks as Work, Study, Personal, or Health: finish project report, review lecture notes, buy groceries, and go for a run. Return one category per task.",3
1,6,C2,Tiny (<0.5B),87.605,2.922,Task Classification,"Categorize my tasks: email my supervisor, prepare for an exam, clean my room, and attend a dental appointment.",3
2,6,C3,Tiny (<0.5B),87.422,0.824,Task Classification,"Classify these activities: submit homework, schedule a team meeting, call my family, and attend a yoga class.",2
3,6,C4,Tiny (<0.5B),81.588,3.138,Task Classification,"Organize these tasks by category: write a research summary, cook dinner, review class slides, and visit the gym.",2
4,6,C5,Tiny (<0.5B),90.059,2.843,Task Classification,"Classify these tasks as Work, Study, Personal, or Health: update a client spreadsheet, study Python, pay rent, and attend a medical checkup.",2
5,6,P1,Tiny (<0.5B),90.829,2.818,Priority Ranking,"I have a report due tomorrow, groceries to buy, and a room to clean. Rank these tasks by priority and explain briefly.",1
6,6,P2,Tiny (<0.5B),89.229,2.869,Priority Ranking,"I have an exam tomorrow, a meeting next week, and laundry to finish. What should I prioritize today?",3
7,6,P3,Tiny (<0.5B),83.817,3.054,Priority Ranking,"My presentation is due tonight, my emails can wait until Friday, and I need to exercise. Rank these tasks.",2
8,6,P4,Tiny (<0.5B),79.432,3.223,Priority Ranking,"I must submit an assignment by 5 PM, call a friend sometime this week, and prepare dinner tonight. Rank these tasks by urgency.",1
9,6,P5,Tiny (<0.5B),89.461,2.862,Priority Ranking,"I need to finish a group project due tomorrow, organize my desk, and prepare for a quiz next Monday. Give me a priority order.",2


PASS: all 125 Stage 2 scored records, prompt text, measurements and score arithmetic.
Grand total: 823 / 1250


### Stage 2 scaling and costs

Reuses Stage 1's exact cost-model contract unchanged: same 6 traffic categories, same 250 input / 500 output tokens-per-request assumption, same AWS g5.xlarge/A10G reference machine at USD 1.006/hour, same 70% utilization and client-gateway assumptions. Cloud throughput assumptions are reused per category from Stage 1 rather than re-guessed, since Models 6-10 sit in the identical five categories as Models 1-5 on the same reference machine: Tiny=60, Small=45, Medium=30, Large=20, Very Large=12 tokens/sec.

Paid-API proxy pricing could not simply be copied from Stage 1: re-checking `https://huggingface.co/inference/models` today (2026-10-22) against the same official source Lin used on 2026-10-21 found that `Qwen/Qwen3-4B-Instruct-2507` is no longer listed on `nscale`. The replacement used for Models 6-9 is `meta-llama/Llama-3.1-8B-Instruct` on `nscale`, verified live today at USD 0.06 input / USD 0.06 output per million tokens. Model 10 reuses Stage 1's still-live `Qwen/Qwen3-8B`/`nscale` quote (USD 0.07 / USD 0.18) — note that this paid 8B proxy is actually *smaller* than the local 12B Mistral-Nemo model it is compared against; no size or quality equivalence is claimed for any of these five rows, consistent with Stage 1's own disclosed proxy limitations. Full evidence is in `llm-test/a7_api_pricing.json`.


In [8]:
pricing_s2 = json.loads((MODULE_DIR / "a7_api_pricing.json").read_text())
api_models_s2 = {int(k): v for k, v in pricing_s2["models"].items()}
cloud_throughput_s2 = {6: 60, 7: 45, 8: 30, 9: 20, 10: 12}
local_cost_df_s2, api_cost_df_s2 = calculate_costs(range(6, 11), model_matrix, cloud_throughput_s2, api_models_s2)
saved_local_s2 = pd.read_csv(RESULTS_DIR / "a7_stage2_local_cost.csv")
saved_api_s2 = pd.read_csv(RESULTS_DIR / "a7_stage2_api_cost.csv")
pd.testing.assert_frame_equal(local_cost_df_s2, saved_local_s2, check_dtype=False)
pd.testing.assert_frame_equal(api_cost_df_s2, saved_api_s2, check_dtype=False)
assert len(local_cost_df_s2) == len(api_cost_df_s2) == 30
assert api_cost_df_s2[["Estimated API Cost", "Total Cost"]].notna().all().all()
with pd.option_context("display.max_rows", 60, "display.max_columns", None):
    display(local_cost_df_s2)
    display(api_cost_df_s2)
print("PASS: 30 local + 30 sourced API scenarios reproduced without writing CSVs.")


,Model Name,Daily Traffic Category,Daily Active Users,Avg Tokens per User,Total Daily Token Load,Per Machine Cost (Cloud),Machines Needed,Total Machine Cost,Cloud Throughput Assumption (tokens/sec),Cost Period
0,HuggingFaceTB/SmolLM-135M-Instruct,Prototype,1000,500,500000,1.006,1,724.32,60,USD/month
1,HuggingFaceTB/SmolLM-135M-Instruct,Early Startup,10000,500,5000000,1.006,2,1448.64,60,USD/month
2,HuggingFaceTB/SmolLM-135M-Instruct,Growing Product,100000,500,50000000,1.006,14,10140.48,60,USD/month
3,HuggingFaceTB/SmolLM-135M-Instruct,Large Platform,1000000,500,500000000,1.006,138,99956.16,60,USD/month
4,HuggingFaceTB/SmolLM-135M-Instruct,Mass Consumer App,10000000,500,5000000000,1.006,1378,998112.96,60,USD/month
5,HuggingFaceTB/SmolLM-135M-Instruct,Global Platform,100000000,500,50000000000,1.006,13779,9980405.28,60,USD/month
6,Qwen/Qwen3-0.6B,Prototype,1000,500,500000,1.006,1,724.32,45,USD/month
7,Qwen/Qwen3-0.6B,Early Startup,10000,500,5000000,1.006,2,1448.64,45,USD/month
8,Qwen/Qwen3-0.6B,Growing Product,100000,500,50000000,1.006,19,13762.08,45,USD/month
9,Qwen/Qwen3-0.6B,Large Platform,1000000,500,500000000,1.006,184,133274.88,45,USD/month


,Local LLM Model Size,Comparable Paid Model Hosted on Hugging Face,Output USD per Token,Daily Traffic Category,Total Token Load,Estimated API Cost,Machines Required (API infrastructure),Total Cost,API Gateway Cost,Pricing Status,Cost Period,Paid Provider,Input USD per 1M tokens,Output USD per 1M tokens,Assumed Input Tokens per Request,Assumed Output Tokens per Request,Total Daily Input Tokens,Model Comparison Note,Price Source URL,Price Checked Date
0,HuggingFaceTB/SmolLM-135M-Instruct,meta-llama/Llama-3.1-8B-Instruct,6.000000e-08,Prototype,500000,1.35,1,37.35,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.06,0.06,250,500,250000,Task-based 8B instruct proxy for a 135M local ...,https://huggingface.co/inference/models?model=...,2026-10-22
1,HuggingFaceTB/SmolLM-135M-Instruct,meta-llama/Llama-3.1-8B-Instruct,6.000000e-08,Early Startup,5000000,13.50,1,49.50,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.06,0.06,250,500,2500000,Task-based 8B instruct proxy for a 135M local ...,https://huggingface.co/inference/models?model=...,2026-10-22
2,HuggingFaceTB/SmolLM-135M-Instruct,meta-llama/Llama-3.1-8B-Instruct,6.000000e-08,Growing Product,50000000,135.00,1,171.00,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.06,0.06,250,500,25000000,Task-based 8B instruct proxy for a 135M local ...,https://huggingface.co/inference/models?model=...,2026-10-22
3,HuggingFaceTB/SmolLM-135M-Instruct,meta-llama/Llama-3.1-8B-Instruct,6.000000e-08,Large Platform,500000000,1350.00,1,1386.00,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.06,0.06,250,500,250000000,Task-based 8B instruct proxy for a 135M local ...,https://huggingface.co/inference/models?model=...,2026-10-22
4,HuggingFaceTB/SmolLM-135M-Instruct,meta-llama/Llama-3.1-8B-Instruct,6.000000e-08,Mass Consumer App,5000000000,13500.00,2,13572.00,72.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.06,0.06,250,500,2500000000,Task-based 8B instruct proxy for a 135M local ...,https://huggingface.co/inference/models?model=...,2026-10-22
5,HuggingFaceTB/SmolLM-135M-Instruct,meta-llama/Llama-3.1-8B-Instruct,6.000000e-08,Global Platform,50000000000,135000.00,12,135432.00,432.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.06,0.06,250,500,25000000000,Task-based 8B instruct proxy for a 135M local ...,https://huggingface.co/inference/models?model=...,2026-10-22
6,Qwen/Qwen3-0.6B,meta-llama/Llama-3.1-8B-Instruct,6.000000e-08,Prototype,500000,1.35,1,37.35,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.06,0.06,250,500,250000,Task-based 8B instruct proxy for a 0.6B local ...,https://huggingface.co/inference/models?model=...,2026-10-22
7,Qwen/Qwen3-0.6B,meta-llama/Llama-3.1-8B-Instruct,6.000000e-08,Early Startup,5000000,13.50,1,49.50,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.06,0.06,250,500,2500000,Task-based 8B instruct proxy for a 0.6B local ...,https://huggingface.co/inference/models?model=...,2026-10-22
8,Qwen/Qwen3-0.6B,meta-llama/Llama-3.1-8B-Instruct,6.000000e-08,Growing Product,50000000,135.00,1,171.00,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.06,0.06,250,500,25000000,Task-based 8B instruct proxy for a 0.6B local ...,https://huggingface.co/inference/models?model=...,2026-10-22
9,Qwen/Qwen3-0.6B,meta-llama/Llama-3.1-8B-Instruct,6.000000e-08,Large Platform,500000000,1350.00,1,1386.00,36.0,Verified HF listing; workload and gateway are ...,USD/month,nscale,0.06,0.06,250,500,250000000,Task-based 8B instruct proxy for a 0.6B local ...,https://huggingface.co/inference/models?model=...,2026-10-22


PASS: 30 local + 30 sourced API scenarios reproduced without writing CSVs.


### Stage 2 handoff

Per-model totals (max 250/model): Model 6 (SmolLM-135M) 47, Model 7 (Qwen3-0.6B) 154, Model 8 (gemma-2-2b-it) 201, Model 9 (Qwen3-4B) 210, Model 10 (Mistral-Nemo-12B, 4-bit) 211. Grand total 823/1250. Ranking matches the shape of Stage 1's: the Tiny model is unusable for this task, the Small model is workable but inconsistent, and Medium/Large/Very Large converge on broadly similar, usable quality.

Two concrete, evidence-based findings worth carrying into Stage 3 and final integration: (1) every model in this slot, including the strongest ones, intermittently gets explicit-deadline priority ranking backwards — Qwen3-0.6B inverted urgency on 3 of 5 Priority Ranking prompts, Mistral-Nemo inverted one; this is a genuine model-reasoning weakness, not a prompting artifact, since the deadline was always stated explicitly in the prompt. (2) `google/gemma-2-2b-it` running on `backend="cpu"` with no MPS path averaged 615.9s/request here versus a fraction of that previously on this same hardware with MPS acceleration (see the prior Prompt Evaluation assignment) — a 2B model is usable quality-wise on CPU but likely impractical for interactive local hosting without GPU/MPS acceleration; this doesn't change its benchmark score, only its latency.

One routing scenario, "Peak Usage Hours," is added separately to `system_design.md`, covering local/API model choice, fallback behavior and separate latency/cost/quality evaluation, using Model 8 and Model 10 from this slot as the local-side examples.
